In [1]:
# ✅ 라이브러리 로드
from pytorch_tabnet.tab_model import TabNetClassifier
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import joblib
import pandas as pd
import numpy as np
import torch
import os
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# ✅ GPU 설정
if torch.cuda.is_available():
    device = torch.device('cuda:2')
    print('🚀 GPU 사용 가능')
else:
    device = torch.device('cpu')
    print('❌ GPU 사용 불가')

print(device)

# ✅ 1. 모델 로드 (파일명 수정)
model_xgb = xgb.Booster()
model_xgb.load_model("3xgb_model.json")  # 파일명 수정

model_lgb = lgb.Booster(model_file="3lgb_model.txt")  # 파일명 수정

model_cat = cb.CatBoostClassifier()
model_cat.load_model("3cat_model.cbm")  # 파일명 수정

meta_model = joblib.load("3meta_model.pkl")  # ✅ Meta Model 로드 (파일명 수정)

# ✅ 2. Test 데이터 로드
test_path = "/root/Public_Storage/madelab_khw/lg_aimers/dataset/test.csv"
df = pd.read_csv(test_path)
df['특정 시술 유형'] = df['특정 시술 유형'].replace('IVF:Unknown:Unknown:Unknown', 'IVF')
df['특정 시술 유형'] = df['특정 시술 유형'].replace('IUI:ICI', 'IUI')

# ✅ 3. 데이터 전처리 함수
import pandas as pd
import joblib
import numpy as np

def pre_process(df, is_train=True):
    
    df = df.drop(columns=['ID'], errors='ignore')  # ✅ ID 제거

    for col in df.columns:
        if df[col].dtype == 'object':
            df[col].fillna('Unknown', inplace=True)
        else:
            if df[col].nunique()<=4:
                df[col].fillna(0, inplace = True)
            else:
                df[col].fillna(df[col].mean(), inplace=True)
                
    # ✅ 저장된 LabelEncoder 불러오기
    encoders = joblib.load("encoders.pkl")  # LabelEncoder 로드
    categorical_columns = df.select_dtypes(include=['object']).columns.tolist()
    
    for col in categorical_columns:
        df[col] = df[col].astype(str)  # 문자열 변환
        df[col] = encoders[col].transform(df[col])  # ✅ 저장된 LabelEncoder 적용

    # ✅ 저장된 MinMaxScaler 불러오기
    scaler = joblib.load("scaler.pkl")  # Scaler 로드
    numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    df[numeric_columns] = scaler.transform(df[numeric_columns])  # ✅ 저장된 Scaler 적용

    return df


X_test_numpy =pre_process(df)
# ✅ Train에서 저장된 컬럼 순서 불러오기
train_columns = joblib.load("train_columns.pkl")

# ✅ Test 데이터 컬럼 순서 맞추기
X_test_numpy = X_test_numpy[train_columns]  


# ✅ 6. 예측 수행 (1st Layer)
test_preds = np.zeros((X_test_numpy.shape[0], 3))

test_preds[:, 0] = model_xgb.predict(xgb.DMatrix(X_test_numpy))  # XGBoost 예측
test_preds[:, 1] = model_lgb.predict(X_test_numpy)  # LightGBM 예측 (넘파이 입력)
test_preds[:, 2] = model_cat.predict_proba(X_test_numpy)[:, 1]  # CatBoost 예측

final_test_preds = (
    
    test_preds[:, 0]*0.4+test_preds[:, 1]*0.3+test_preds[:, 2]*0.3
    
)



🚀 GPU 사용 가능
cuda:2


In [2]:
test_df = pd.read_csv('/root/Public_Storage/madelab_khw/lg_aimers/dataset/sample_submission.csv')


test_df["probability"] = final_test_preds
test_df.to_csv("last5_test_with_predictions.csv", index=False)
print("예측 결과가 test_with_predictions.csv 파일로 저장되었습니다.")

예측 결과가 test_with_predictions.csv 파일로 저장되었습니다.
